# Урок 1. Из чего состоит игра

В этом уроке мы разберем самые базовые идеи, которые есть почти в любой игре:

- игровое состояние;
- кадр;
- игровой цикл;
- частота кадров, или FPS;
- ввод с клавиатуры;
- движение и столкновения.

Мы будем использовать нашу маленькую библиотеку, но пока не будем говорить про железо, экраны и микроконтроллеры. Сегодня игра запускается на компьютере.

## 1. Импорт библиотеки и настройки игры

Импортируем несколько вещей из нашей библиотеки и зададим размер игрового поля:

- `GameObject` - простой игровой объект;
- цвета;
- размер игрового поля.

In [1]:
from api import GameObject, Keys, BLACK, GREEN, RED, SKY, WHITE

WIDTH = 160
HEIGHT = 120

print("Размер игрового поля:", WIDTH, "x", HEIGHT)

Размер игрового экрана: 160 x 120


## 2. Что такое игра

Игра - это программа, которая много раз повторяет три действия:

1. читает ввод игрока;
2. обновляет мир игры;
3. рисует новый кадр.

Например, если герой идет вправо, игра не телепортирует его сразу в конец экрана. Она каждый кадр чуть-чуть меняет координату `x`.

In [2]:
player = GameObject(x=10, y=80, w=12, h=12, color=GREEN)

print("x до движения:", player.x)
player.vx = 3
player.move()
print("x после одного кадра:", player.x)

x до движения: 10
x после одного кадра: 13


У `GameObject` есть:

- `x`, `y` - положение;
- `w`, `h` - размер;
- `vx`, `vy` - скорость по горизонтали и вертикали.

Метод `move()` прибавляет скорость к координатам.

In [3]:
player = GameObject(10, 80, 12, 12, GREEN)
player.vx = 3

for frame in range(1, 6):
    player.move()
    print("кадр", frame, "x =", player.x)

кадр 1 x = 13
кадр 2 x = 16
кадр 3 x = 19
кадр 4 x = 22
кадр 5 x = 25


## 3. Кадр

Кадр - это одна картинка игры.

Если игра работает в 30 FPS, это значит, что за одну секунду она успевает показать примерно 30 кадров.

Посчитаем, сколько миллисекунд есть на один кадр.

In [4]:
for fps in [15, 30, 60]:
    ms_per_frame = 1000 / fps
    print(f"{fps} FPS -> примерно {ms_per_frame:.1f} мс на кадр")

15 FPS -> примерно 66.7 мс на кадр
30 FPS -> примерно 33.3 мс на кадр
60 FPS -> примерно 16.7 мс на кадр


Чем больше FPS, тем меньше времени есть на обновление и рисование одного кадра.

Для маленьких игр 30 FPS часто достаточно: движение уже выглядит плавно, а компьютеру легче успевать.

## 4. Игровой цикл

Удобно договориться, что у игры есть два главных метода:

- `update(keys)` - изменить состояние игры;
- `draw(gfx)` - нарисовать состояние игры.

Один кадр игры обычно выглядит так:

```python
keys = прочитать_кнопки()
game.update(keys)   # изменить состояние игры
game.draw(gfx)      # нарисовать новое состояние
подождать_до_следующего_кадра()
```

Потом эти действия повторяются снова и снова. В этом и есть игровой цикл.

В настоящем окне этот цикл будет запускать `run(...)`, а сейчас мы сделаем несколько кадров вручную, чтобы увидеть порядок действий.

In [5]:
class TinyGame:
    def __init__(self):
        self.player = GameObject(10, 80, 12, 12, GREEN)

    def update(self, keys):
        self.player.vx = 1
        self.player.move()

    def draw(self, gfx):
        gfx.clear(SKY)
        self.player.draw(gfx)
        gfx.present()

game = TinyGame()
keys = Keys()

for frame in range(5):
    print("кадр", frame)
    print("  1. update: меняем координаты")
    game.update(keys)
    print("     x игрока =", game.player.x)
    print("  2. draw: здесь окно перерисовало бы экран")

кадр 0 x игрока = 11
кадр 1 x игрока = 12
кадр 2 x игрока = 13
кадр 3 x игрока = 14
кадр 4 x игрока = 15


В примере выше мы не рисовали окно, но порядок уже такой же, как в настоящей игре.

Каждый кадр:

1. игра обновляет координаты в `update`;
2. потом должна нарисовать новую картинку в `draw`.

Когда мы позже вызовем `run(CatchGiftGameWithDraw())`, функция `run` будет делать это автоматически примерно 30 раз в секунду.

## 5. Ввод игрока

В игре нельзя просто двигать героя всегда вправо. Нужно смотреть, какие кнопки нажаты.

В нашей библиотеке объект `keys` хранит состояние направлений и кнопок.

Это не команды игры, а просто ввод:

- `keys.left` - нажато направление влево;
- `keys.right` - нажато направление вправо;
- `keys.up` - нажато направление вверх;
- `keys.down` - нажато направление вниз;
- `keys.button_a` - нажата первая кнопка действия;
- `keys.button_b` - нажата вторая кнопка действия;
- `keys.button_menu` - нажата кнопка меню.

Смысл кнопок придумывает сама игра. В одной игре `button_a` может быть ударом, в другой - прыжком, в третьей - выбором пункта меню.

In [6]:
player = GameObject(60, 80, 12, 12, GREEN)
keys = Keys()

keys.right = True

if keys.left:
    player.vx = -2
elif keys.right:
    player.vx = 2
else:
    player.vx = 0

player.move()
print("Игрок сдвинулся в x =", player.x)

Игрок сдвинулся в x = 62


## 6. Границы экрана

Если каждый кадр двигать объект, он может улететь за экран.

Метод `keep_inside(width, height)` удерживает объект внутри прямоугольника.

In [7]:
player = GameObject(150, 80, 20, 12, GREEN)
print("до:", player.x)

player.keep_inside(WIDTH, HEIGHT)
print("после:", player.x)

до: 150
после: 140


## 7. Столкновения

Столкновение - это ситуация, когда два объекта пересекаются.

Например, игрок поймал подарок.

In [8]:
player = GameObject(50, 90, 20, 8, GREEN)
gift = GameObject(55, 88, 8, 8, RED)

if player.overlaps(gift):
    print("Подарок пойман!")
else:
    print("Пока не пойман")

Подарок пойман!


## 8. Мини-игра без окна: поймай подарок

Сначала сделаем игру и проверим ее несколькими кадрами без рисования окна. Это хороший способ искать ошибки в логике.

In [9]:
class CatchGiftGame:
    def __init__(self):
        self.player = GameObject(WIDTH // 2 - 10, HEIGHT - 12, 20, 6, GREEN)
        self.gift = GameObject(20, 0, 8, 8, RED)
        self.score = 0
        self.missed = 0

    def update(self, keys):
        self.player.vx = 0
        if keys.left:
            self.player.vx = -3
        if keys.right:
            self.player.vx = 3

        self.player.move()
        self.player.keep_inside(WIDTH, HEIGHT)

        self.gift.vy = 2
        self.gift.move()

        if self.player.overlaps(self.gift):
            self.score += 1
            self.reset_gift()
        elif self.gift.y > HEIGHT:
            self.missed += 1
            self.reset_gift()

    def reset_gift(self):
        self.gift.x = (self.gift.x * 37 + 23) % (WIDTH - self.gift.w)
        self.gift.y = 0
        self.gift.vy = 0

game = CatchGiftGame()
keys = Keys()
keys.right = True

for frame in range(10):
    game.update(keys)
    print("кадр", frame, "игрок x =", game.player.x, "подарок y =", game.gift.y, "счет =", game.score)

кадр 0 игрок x = 73 подарок y = 2 счет = 0
кадр 1 игрок x = 76 подарок y = 4 счет = 0
кадр 2 игрок x = 79 подарок y = 6 счет = 0
кадр 3 игрок x = 82 подарок y = 8 счет = 0
кадр 4 игрок x = 85 подарок y = 10 счет = 0
кадр 5 игрок x = 88 подарок y = 12 счет = 0
кадр 6 игрок x = 91 подарок y = 14 счет = 0
кадр 7 игрок x = 94 подарок y = 16 счет = 0
кадр 8 игрок x = 97 подарок y = 18 счет = 0
кадр 9 игрок x = 100 подарок y = 20 счет = 0


## 9. Рисование

Метод `draw(gfx)` получает объект для рисования. Мы не думаем, как именно он рисует пиксели. Мы просто говорим:

- очистить экран;
- нарисовать игрока;
- нарисовать подарок;
- показать кадр.

In [10]:
class CatchGiftGameWithDraw(CatchGiftGame):
    def draw(self, gfx):
        gfx.clear(SKY)
        self.player.draw(gfx)
        self.gift.draw(gfx)
        self.draw_score(gfx)
        gfx.present()

    def draw_score(self, gfx):
        for i in range(self.score % 10):
            gfx.rect(2 + i * 4, 2, 3, 3, WHITE)
        for i in range(self.missed % 10):
            gfx.rect(2 + i * 4, 7, 3, 3, BLACK)

## 10. Запуск игры в окне

Следующая ячейка откроет окно с игрой.

Управление:

- стрелка влево или `A` - влево;
- стрелка вправо или `D` - вправо.

Когда закончишь играть, закрой окно. После этого ячейка завершится.

In [11]:
from backends.tkinter_backend import run

run(CatchGiftGameWithDraw())

## 11. Задания

Попробуй изменить игру:

1. Сделай подарок быстрее.
2. Сделай игрока шире или уже.
3. Добавь второй падающий объект другого цвета.
4. Сделай так, чтобы после 5 пропущенных подарков игра печатала `GAME OVER`.
5. Сделай счет не квадратиками, а текстом через `gfx.text(...)`.

Главная идея урока: игра - это не магия. Это состояние, которое меняется много раз в секунду, и картинка, которая рисуется после каждого изменения.